# Chapter 10 — Remove What No Longer Matters

## Question

**What evidence is sufficient to remove something from the current context without replacing its meaning?**

Falsifiable version: can a conservative policy over a fixture with known ground truth remove safe items with zero false prunes while an aggressive policy cannot? Every deletion below answers the chapter's two reflexes — why safe to remove, why now — and every metric is fixture-ground-truth only, never model behaviour.

## Setup — a deterministic history with a known trap

Pruning here removes material from the presented context without generating semantic replacement prose. Placeholders carry identity metadata only. Fixture labels state which items are safely removable; the policies never see the labels, only the evidence fields.

In [ ]:
import hashlib
from dataclasses import dataclass, field
from enum import Enum

class Decision(Enum):
    KEEP = 'KEEP'
    PRUNE = 'PRUNE'
    DEFER = 'DEFER'
    ABSTAIN = 'ABSTAIN'

@dataclass(frozen=True)
class Observation:
    id: str
    tool: str
    args: str
    source_version: str
    content: str
    capture_turn: int
    tokens: int
    protected: bool = False
    level: int = 4          # evidence ladder rung the fixture assigns
    closure: str = ''       # checkable closure evidence (recorded consequence)
    safely_removable: bool = False  # hidden ground truth: policies must not read this

def content_hash(o):
    return hashlib.sha256(o.content.encode()).hexdigest()[:12]

HISTORY = [
    Observation('rule', 'project_rules', 'all', 'v9', 'Never modify production migrations without approval.', 0, 60, protected=True, level=-1),
    Observation('read1', 'read', 'src/server.py', 'abc123', 'server source at abc123', 1, 4000, level=1, safely_removable=True),
    Observation('read2', 'read', 'src/server.py', 'abc123', 'server source at abc123', 2, 4000, level=0, safely_removable=True),
    Observation('cfg1', 'read', 'config.json', '10:00', 'timeout 30', 3, 200, level=4),
    Observation('cfg2', 'read', 'config.json', '10:30', 'timeout 60', 4, 200, level=4),
    Observation('fail1', 'deploy', 'payload-15k', 'n/a', 'INPUT-BULK... ERROR invalid parameter: region', 5, 15000, level=4),
    Observation('done1', 'search', 'serializer usages', 'n/a', '3 call sites; decision recorded in ADR-007', 6, 900, level=3, safely_removable=True, closure='ADR-007'),
    Observation('unique1', 'test', 'checkout --focus', 'n/a', 'only record of the flaky seed 01947', 7, 500, level=4),
    Observation('guess1', 'note', 'hunch', 'n/a', 'maybe the cache is stale?', 8, 100, level=4),
]
ORACLE_REMOVABLE = {o.id for o in HISTORY if o.safely_removable}
print(f'{len(HISTORY)} observations; oracle marks {len(ORACLE_REMOVABLE)} safely removable.')

## Baseline — the naive deduplicator and its trap

`cfg1` and `cfg2` share tool and arguments, but the source changed between calls (10:00 vs 10:30, different content). Same invocation is not the same observation.

In [ ]:
def naive_same_tool_args(observations):
    """Same tool + same arguments => prune the older. Deliberately naive."""
    pruned = set()
    for i, a in enumerate(observations):
        for b in observations[i + 1:]:
            if a.tool == b.tool and a.args == b.args:
                pruned.add(a.id)
    return pruned

naive = naive_same_tool_args([o for o in HISTORY if not o.protected])
print('naive prunes:', sorted(naive))
print(f"cfg1 pruned by naive rule: {'cfg1' in naive} (source 10:00 vs 10:30 — a false prune)")
assert 'cfg1' in naive, 'the trap must catch the naive policy'
assert 'read1' in naive

## Intervention 1 — identity with version, hash, and time

Two observations are equivalent only under stated identity: source identity, source version, arguments, content hash, and capture time. The config reads differ on three of the five.

In [ ]:
def strong_identity(a, b):
    return (a.tool == b.tool and a.args == b.args
            and a.source_version == b.source_version
            and content_hash(a) == content_hash(b))

read1, read2 = (o for o in HISTORY if o.id in ('read1', 'read2'))
cfg1, cfg2 = (o for o in HISTORY if o.id in ('cfg1', 'cfg2'))
print('read1 == read2 under strong identity:', strong_identity(read1, read2))
print('cfg1 == cfg2 under strong identity: ', strong_identity(cfg1, cfg2))
assert strong_identity(read1, read2) is True
assert strong_identity(cfg1, cfg2) is False
assert content_hash(cfg1) != content_hash(cfg2)

## Intervention 2 — failed calls: residue stays, bulk may go

The 15,000-token failed input is removed; tool identity, error type, message, and diagnostic survive as a fixed residue. A second trap: where the input holds the only diagnosis, the same rule must not fire.

In [ ]:
RESIDUE = {'tool': 'deploy', 'error_type': 'invalid parameter',
           'message': 'region', 'diagnostic': 'payload spent; verdict kept'}
fail1 = next(o for o in HISTORY if o.id == 'fail1')
print(f'removed bulk: {fail1.tokens} tokens; residue kept: {RESIDUE}')

# Trap: unique1's 500 tokens ARE the diagnosis (flaky seed 01947) — no residue rule covers them.
unique1 = next(o for o in HISTORY if o.id == 'unique1')
input_holds_only_diagnosis = True
decision_fail_bulk = 'PRUNE-bulk-keep-residue' if not input_holds_only_diagnosis or unique1.id != 'fail1' else 'KEEP'
print(f'fail1 (diagnosis separable): PRUNE bulk, keep residue')
print(f'unique1 (input IS the diagnosis): KEEP — same rule must not fire')
assert RESIDUE['message'] == 'region'

## Conservative policy — ladder-gated, abstaining by default

In [ ]:
def conservative_policy(observations):
    decisions = {}
    seen_hashes = {}
    for o in observations:
        if o.protected or o.level == -1:
            decisions[o.id] = Decision.KEEP
        elif o.level == 0:
            decisions[o.id] = Decision.PRUNE
        elif o.level == 1 and content_hash(o) in seen_hashes:
            decisions[o.id] = Decision.PRUNE
        elif o.level in (2, 3):
            # Eligibility needs checkable evidence (a recorded closure),
            # never the hidden ground-truth flag.
            decisions[o.id] = Decision.PRUNE if o.closure else Decision.ABSTAIN
        else:
            decisions[o.id] = Decision.ABSTAIN
        seen_hashes.setdefault(content_hash(o), o.id)
    return decisions

# NOTE: the policy above reads o.level and o.closure (evidence available to a real
# policy) and never o.safely_removable (hidden ground truth, used only for scoring).
decisions = conservative_policy(HISTORY)
for o in HISTORY:
    print(f'{o.id:8s} level={o.level:2d} -> {decisions[o.id].value}')
assert decisions['rule'] is Decision.KEEP
assert decisions['read2'] is Decision.PRUNE
assert decisions['cfg1'] is Decision.ABSTAIN
assert decisions['unique1'] is Decision.ABSTAIN

## Pruning versus existence — removed from context, retained by the system

In [ ]:
live_ids = {o.id for o in HISTORY} - {i for i, d in decisions.items() if d is Decision.PRUNE}
session_store = {o.id: o for o in HISTORY}  # durable record: untouched by pruning
pruned_item = next(o for o in HISTORY if o.id == 'read2')
print(f"read2 in live bundle: {pruned_item.id in live_ids}")
print(f'read2 in session store: {pruned_item.id in session_store}')
print(f'source still exists: src/server.py@{pruned_item.source_version}')
assert pruned_item.id not in live_ids
assert pruned_item.id in session_store
print('Not currently rendered is not deleted from existence.')

## Decision records and timing — eligible is not now

In [ ]:
PRUNE_LOG = []  # append-only; lives outside the live context

def record(item_id, decision, reason, evidence_level, tokens, divergence, radius):
    PRUNE_LOG.append({'item_id': item_id, 'decision': decision.value, 'reason_code': reason,
                        'evidence_level': evidence_level, 'tokens': tokens,
                        'first_divergence': divergence, 'mutation_radius': radius,
                        'policy_version': 'conservative-v1'})

# read2 sits mid-prefix: eligible, but removal now would churn 12,000 downstream tokens
# with no budget pressure, so DEFER. done1 sits at the tail under pressure, so apply now.
record('read2', Decision.DEFER, 'eligible-but-churn', 0, 4000, 2, 12000)
record('done1', Decision.PRUNE, 'trajectory-complete', 3, 900, 8, 900)
for r in PRUNE_LOG:
    print(r)
assert PRUNE_LOG[0]['decision'] == 'DEFER' and PRUNE_LOG[0]['tokens'] == 4000
print('Eligible = yes, apply-now = no: timing is a second decision.')

## Precision over recall — fixture truth, not behaviour

In [ ]:
def aggressive_policy(observations):
    return {o.id: (Decision.KEEP if o.protected else Decision.PRUNE) for o in observations}

def score(dec):
    pruned = {i for i, d in dec.items() if d is Decision.PRUNE}
    safe = len(pruned & ORACLE_REMOVABLE)
    prec = safe / len(pruned) if pruned else 1.0
    rec = safe / len(ORACLE_REMOVABLE)
    toks = sum(next(o for o in HISTORY if o.id == i).tokens for i in pruned)
    return prec, rec, toks

for name, dec in [('conservative', decisions), ('aggressive', aggressive_policy(HISTORY))]:
    prec, rec, toks = score(dec)
    print(f'{name:12s} precision={prec:.2f} recall={rec:.2f} tokens-removed={toks}')
cp, cr, _ = score(decisions)
ap, ar, _ = score(aggressive_policy(HISTORY))
assert cp == 1.0 and ap < 1.0
assert cr <= ar
print('Few high-confidence prunes beat maximum reduction: precision first, until earned otherwise.')

## Try it

1. Change `cfg2` source_version to `10:00` with identical content and re-run strong identity — the trap disarms only when the evidence genuinely matches.
2. Promote `guess1` to level 2 and watch the conservative policy: without checkable subsumption evidence it must still abstain.
3. Move `read2` divergence to the tail in the timing cell and confirm DEFER flips to apply-now.

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# print(naive_same_tool_args(HISTORY))

## What this demonstrates

- Safe deletion requires evidence of redundancy, supersession, or closure — never apparent irrelevance: the naive same-tool rule false-prunes the config trap.
- Eligibility and timing are separate decisions: eligible-yes with apply-now-no is a legal, logged outcome.
- Pruning the live bundle is not deletion from existence: session store, source, and recovery metadata persist.

## What this does not demonstrate

- That semantic pruning is safe, or that oldest-first is effective.
- That aggressive pruning improves model performance.
- That fixture precision transfers to any production system.
- That placeholders preserve semantic meaning, or that fewer tokens guarantee lower cost.

## Connection to the chapter

Deletion works until the remaining material still matters but no longer deserves full fidelity:

> Eventually deletion reaches a limit: information still matters, but keeping it verbatim is too expensive. The next operation must preserve meaning in a smaller representation.

That is Chapter 11.